# Notebook 04 — Features Lingüísticas Interpretables

## Objetivo

Extraer un conjunto de **features lingüísticas explícitas** por artículo y entrenar un
clasificador de **Regresión Logística** sobre ellas (Experimento 2 del informe).

A diferencia de los modelos BoW/TF-IDF del Notebook 03, aquí el objetivo **no es maximizar
performance** sino la **interpretabilidad de coeficientes**: un coeficiente alto en
`ratio_exclamacion` se lee directamente como "más exclamaciones → más probable fake".

> **Estado: scaffold.** La estructura del experimento está definida; las celdas de código
> están pendientes de implementar. Antes de completarlo:
> - Agregar `vaderSentiment` a `pyproject.toml` (VADER no está instalado todavía).
> - (Opcional) Crear `src/nlp/features.py` con la extracción reutilizable y cachearla en `data/processed/`.
> - spaCy `en_core_web_sm` ya está disponible: POS y NER son suficientes para estas features
>   (no se usan word vectors, por eso no hace falta `en_core_web_lg`).
>
> Referencia metodológica: [ADR Experimento 2](../docs/adr/experimento-02-features-linguisticas.md).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from sklearn.linear_model import LogisticRegression

from nlp.paths import DATA_PROCESSED, RESULTS_METRICS
from nlp.metrics import compute_metrics, metrics_to_row

# TODO: dependencias de extracción de features
# import spacy  # en_core_web_sm
# from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

## 1. Carga de splits (subconjunto político)

In [ ]:
train = pd.read_csv(DATA_PROCESSED / "politics_train.csv")
val = pd.read_csv(DATA_PROCESSED / "politics_val.csv")
test = pd.read_csv(DATA_PROCESSED / "politics_test.csv")
train.shape, val.shape, test.shape

## 2. Extracción de features lingüísticas

Vector de **8 features explícitas** por artículo (ver ADR Exp. 2):

| Feature | Herramienta | Señal esperada |
| :------ | :---------- | :------------- |
| `ratio_exclamacion` | regex / conteo | `!` por oración (sensacionalismo) |
| `ratio_mayusculas`  | regex | proporción de palabras en MAYÚSCULAS |
| `long_oracion_prom` | spaCy (segmentación) | tokens por oración |
| `ratio_adj_sust`    | spaCy POS | adjetivos / sustantivos |
| `sentimiento_vader` | VADER | score compuesto |
| `densidad_ner`      | spaCy NER | entidades por oración |
| `freq_url`          | conteo de `[URL]` | enlaces externos |
| `freq_pronombres`   | regex / POS | pronombres 1.ª/2.ª persona |

In [ ]:
# nlp = spacy.load("en_core_web_sm")
# vader = SentimentIntensityAnalyzer()


def extract_features(text: str) -> dict:
    """Extrae las 8 features lingüísticas de un artículo.

    TODO: implementar cada feature.
      ratio_exclamacion  -> signos '!' por oración
      ratio_mayusculas   -> proporción de palabras en MAYÚSCULAS
      long_oracion_prom  -> tokens por oración (spaCy)
      ratio_adj_sust     -> adjetivos / sustantivos (spaCy POS)
      sentimiento_vader  -> score compuesto VADER
      densidad_ner       -> entidades por oración (spaCy NER)
      freq_url           -> ocurrencias del token [URL]
      freq_pronombres    -> pronombres 1.ra/2.da persona
    """
    raise NotImplementedError


# TODO: aplicar extract_features a train/val/test y armar matrices X_train/X_val/X_test.
# Sugerencia: cachear el resultado en data/processed/ porque spaCy es costoso.

## 3. Clasificador logístico sobre las 8 features

Regresión Logística (no Random Forest/SVM): el objetivo es **leer los coeficientes**, no
maximizar F2. Métrica principal igual que el resto del TP: **F2 de la clase fake**.

In [ ]:
# TODO: entrenar LogisticRegression sobre las features, seleccionar C con validación (F2 fake).
# clf = LogisticRegression(max_iter=1000, random_state=42)
# clf.fit(X_train, train["label"])

## 4. Coeficientes por feature (interpretabilidad)

Tabla de coeficientes ordenados por magnitud: positivo → asociado a fake, negativo → real.
Es el **resultado lingüístico central** del experimento.

In [ ]:
# TODO: construir DataFrame feature -> coeficiente y ordenar por |coef|.

## 5. Sub-experimento — título vs. cuerpo vs. combinado (Hipótesis 2)

Extraer features y entrenar el clasificador en tres condiciones (`title_text`, `body_text`,
`full_text`) y comparar F2. Valida que el cuerpo aporta más patrones que el titular solo.

In [ ]:
# TODO: para field in ["title_text", "body_text", "full_text"]:
#   extraer features, entrenar, evaluar en val; tabla comparativa de F2.

## 6. Evaluación final en test y persistencia

Evaluar la mejor configuración **una sola vez** en test y guardar las métricas. Para que el
modelo aparezca en la tabla unificada habrá que sumarlo en `nlp.metrics.consolidate_results`.

In [ ]:
# TODO:
# m = compute_metrics(test["label"], y_pred, y_proba)
# row = metrics_to_row(m, {"model": "linguistic_features_lr", "dataset_scope": "politics", "split": "test"})
# pd.DataFrame([row]).to_csv(RESULTS_METRICS / "linguistic_features_results.csv", index=False)

## Conclusiones

_TODO: ¿qué features resultaron más discriminativas? ¿Un modelo de 8 features alcanza un F2
competitivo con el baseline del Notebook 03? ¿Qué campo (título/cuerpo/combinado) concentra
la señal lingüística?_